# Cloud Platforms & AI Infrastructure

**Track:** Foundational Infrastructure  
**Toolchain:** AWS · GCP · Azure · Terraform · Cloud CLI  
**Objective:** Understand where AI workloads actually run in production, how to choose between managed services, and how to provision infrastructure without clicking through a web console.

---

## 🧠 Why Cloud Platforms Before Everything Else?

In the previous notebook, you learned Docker, Kubernetes, and Linux. But WHERE do those containers actually run? On **cloud platforms.**

**Real-world reality:**  
99% of production AI systems run on one of three clouds: AWS, Google Cloud (GCP), or Microsoft Azure. If you've never used a cloud console, CLI, or provisioning tool, you'll be lost on day one of any AI engineering job.

### 🤔 Why Not Just Use My Own Computer?

| Your Laptop | Cloud |
|-------------|-------|
| 1 GPU (if you're lucky) | 1000+ GPUs on demand |
| 16-64 GB RAM | Up to 24 TB RAM per instance |
| Training takes days | Training takes hours (parallelized) |
| If your laptop dies, everything is lost | Data replicated across 3+ regions |
| You pay even when idle | Pay only when running (with proper setup) |

### What This Notebook Covers

```
1. The Big Three clouds → AWS, GCP, Azure for AI
2. Managed AI Services → SageMaker, Vertex AI, Azure AI Studio
3. GPU Instances → Types, costs, when to use each
4. Infrastructure as Code → Terraform for reproducible infra
5. Cloud Cost Management → Don't go bankrupt training models
```

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** The biggest difference between a junior and senior AI engineer isn't knowing more algorithms; it's understanding GPU economics. If you waste $10k of cloud compute, it doesn't matter how good your model is.

**Mental Model:** Cloud platforms (AWS/GCP/Azure) are utility companies. You rent GPUs by the second. Infrastructure as Code (Terraform) is the blueprint that ensures you deploy exactly the server you intended.

**Common Junior Pitfall:** Manually clicking through cloud UX consoles. Seniors script everything (IaC) so infrastructure is version-controlled and tearing down a $30/hr GPU cluster is just one command.

---


## 1 · The Big Three: AWS vs GCP vs Azure

### 🤔 Which Cloud Should I Choose?

This is one of the most common questions. Here's the honest answer:

| Cloud | AI Strength | Market Share | Best For |
|-------|------------|-------------|----------|
| **AWS** | Broadest service catalog, SageMaker | ~32% | Enterprise, startups, most jobs |
| **GCP** | Best AI/ML tools (TPUs, Vertex AI, Gemini) | ~12% | AI-first companies, research |
| **Azure** | OpenAI integration, enterprise compliance | ~23% | Enterprise with Microsoft stack |

### 🤔 Do I Need to Learn All Three?

**No.** Learn ONE deeply and understand the concepts. The concepts transfer:
- AWS S3 ≈ GCP Cloud Storage ≈ Azure Blob Storage (object storage)
- AWS EC2 ≈ GCP Compute Engine ≈ Azure Virtual Machines (compute)
- AWS SageMaker ≈ GCP Vertex AI ≈ Azure AI Studio (managed ML)

**Recommendation for beginners:** Start with AWS (most jobs list it) or GCP (best AI tools).

### The Cloud Service Pyramid

```
        [AI-Specific Services]        ← SageMaker, Vertex AI, Bedrock
       /  Highest abstraction  \
      [Managed Compute]               ← ECS, Cloud Run, AKS
     /  Medium abstraction  \
    [Raw Infrastructure]              ← EC2, GCE, VMs + GPUs
   /   Lowest abstraction  \
  [Storage & Networking]              ← S3, VPC, Load Balancers
```

**Rule of thumb:** Start at the TOP of the pyramid (managed services). Only go lower when you need more control or cost savings.

In [ ]:
# Cell 1 — Cloud service mapping across providers
#
# This cell helps you translate between cloud providers.
# If a job posting says "experience with S3" and you only know GCP,
# you can say "I used Cloud Storage, which is equivalent."

cloud_mapping = {
    'Object Storage': {'AWS': 'S3', 'GCP': 'Cloud Storage', 'Azure': 'Blob Storage'},
    'Compute (VMs)': {'AWS': 'EC2', 'GCP': 'Compute Engine', 'Azure': 'Virtual Machines'},
    'Kubernetes': {'AWS': 'EKS', 'GCP': 'GKE', 'Azure': 'AKS'},
    'Serverless Containers': {'AWS': 'Fargate', 'GCP': 'Cloud Run', 'Azure': 'Container Apps'},
    'ML Platform': {'AWS': 'SageMaker', 'GCP': 'Vertex AI', 'Azure': 'Azure AI Studio'},
    'LLM API': {'AWS': 'Bedrock', 'GCP': 'Gemini API', 'Azure': 'Azure OpenAI'},
    'GPU Instances': {'AWS': 'P5/P4d (A100, H100)', 'GCP': 'A3/A2 (H100, A100)', 'Azure': 'ND H100 v5'},
    'Data Warehouse': {'AWS': 'Redshift', 'GCP': 'BigQuery', 'Azure': 'Synapse'},
    'Secrets Manager': {'AWS': 'Secrets Manager', 'GCP': 'Secret Manager', 'Azure': 'Key Vault'},
    'Container Registry': {'AWS': 'ECR', 'GCP': 'Artifact Registry', 'Azure': 'ACR'},
    'IAM': {'AWS': 'IAM', 'GCP': 'IAM', 'Azure': 'Entra ID'},
    'Monitoring': {'AWS': 'CloudWatch', 'GCP': 'Cloud Monitoring', 'Azure': 'Monitor'},
}

print('☁️ Cloud Service Translation Table')
print(f'{"Service":<25} {"AWS":<25} {"GCP":<25} {"Azure":<25}')
print('=' * 100)
for service, providers in cloud_mapping.items():
    print(f'{service:<25} {providers["AWS"]:<25} {providers["GCP"]:<25} {providers["Azure"]:<25}')

print('\n💡 Key insight: Once you understand the CONCEPT, switching clouds is just learning new names.')

---

## 2 · GPU Instances: The Heart of AI Infrastructure

### 🤔 What GPU Do I Need?

GPUs are the most expensive part of AI infrastructure. Choosing wrong wastes thousands of dollars.

| GPU | VRAM | Best For | AWS Cost/hr | When to Use |
|-----|------|---------|-------------|-------------|
| **T4** | 16 GB | Small inference, fine-tuning small models | ~$0.50 | Budget inference, dev/test |
| **A10G** | 24 GB | Medium inference, LoRA fine-tuning | ~$1.00 | Single 7B model inference |
| **L4** | 24 GB | Inference (latest gen, power efficient) | ~$0.80 | Cost-efficient inference |
| **A100 40GB** | 40 GB | Training, large model inference | ~$3.50 | Fine-tuning 13B-30B models |
| **A100 80GB** | 80 GB | Large model training | ~$5.00 | Fine-tuning 30B-70B models |
| **H100** | 80 GB | Fastest training, large inference | ~$8.00 | Production training at scale |
| **H200** | 141 GB | Largest models, longest context | ~$12.00 | 70B+ models, cutting edge |

### 🤔 What About Spot/Preemptible Instances?

Cloud providers sell UNUSED GPU capacity at 60-90% discount. The catch: they can take it back with 2 minutes notice.

| Instance Type | Discount | Risk | Use For |
|--------------|----------|------|--------|
| **On-demand** | 0% | None | Production inference |
| **Spot/Preemptible** | 60-90% off | Can be terminated anytime | Training with checkpoints |
| **Reserved** | 30-60% off | Commit to 1-3 years | Steady production workloads |

**Production tip:** ALWAYS use checkpointing with spot instances. Save your model every N steps so you can resume if interrupted.

In [ ]:
# Cell 2 — GPU Cost Calculator
#
# This tool helps you estimate the cost of training and inference.
# In real life, GPU costs are the biggest expense for AI teams.

gpu_catalog = {
    'T4':        {'vram_gb': 16,  'cost_hr': 0.50,  'tflops_fp16': 65},
    'A10G':      {'vram_gb': 24,  'cost_hr': 1.00,  'tflops_fp16': 125},
    'L4':        {'vram_gb': 24,  'cost_hr': 0.80,  'tflops_fp16': 121},
    'A100_40GB': {'vram_gb': 40,  'cost_hr': 3.50,  'tflops_fp16': 312},
    'A100_80GB': {'vram_gb': 80,  'cost_hr': 5.00,  'tflops_fp16': 312},
    'H100':      {'vram_gb': 80,  'cost_hr': 8.00,  'tflops_fp16': 990},
    'H200':      {'vram_gb': 141, 'cost_hr': 12.00, 'tflops_fp16': 990},
}

def estimate_training_cost(model_params_b, gpu_name, hours_estimate):
    """Estimate cost to train/fine-tune a model."""
    gpu = gpu_catalog[gpu_name]
    # Rule of thumb: model needs ~2x params in GB for training (weights + gradients + optimizer)
    min_vram_needed = model_params_b * 2  # in GB, for fp16
    gpus_needed = max(1, int(min_vram_needed / gpu['vram_gb']) + 1)
    total_cost = gpus_needed * gpu['cost_hr'] * hours_estimate
    spot_cost = total_cost * 0.3  # ~70% discount
    return {
        'gpu': gpu_name, 'gpus_needed': gpus_needed,
        'on_demand_cost': f'${total_cost:,.0f}',
        'spot_cost': f'${spot_cost:,.0f}',
        'vram_per_gpu': f'{gpu["vram_gb"]} GB',
    }

# Common scenarios
scenarios = [
    ('LoRA fine-tune 7B model',    7,  'A10G',      4),
    ('LoRA fine-tune 13B model',   13, 'A100_40GB', 6),
    ('Full fine-tune 7B model',    7,  'A100_80GB', 24),
    ('Full fine-tune 70B model',   70, 'H100',      72),
]

print('💰 GPU Cost Estimates for Common AI Tasks')
print('=' * 80)
for name, params, gpu, hours in scenarios:
    result = estimate_training_cost(params, gpu, hours)
    print(f'  {name:<30} | {result["gpus_needed"]} x {gpu:<12} | '
          f'On-demand: {result["on_demand_cost"]:>8} | Spot: {result["spot_cost"]:>8}')

print('\n💡 Spot instances save 60-90% but require checkpoint-based training.')
print('   Always save checkpoints every 30-60 minutes when using spot!')

---

## 3 · Managed AI Services: Don't Build What You Can Buy

### 🤔 Self-Hosted vs Managed: When to Use Each?

| Criterion | Self-Hosted (K8s + vLLM) | Managed (SageMaker / Vertex AI) |
|-----------|--------------------------|----------------------------------|
| Setup time | Days-weeks | Minutes-hours |
| Customization | Full control | Limited to provider's options |
| Scaling | You manage autoscaling | Auto-scaling built-in |
| Cost at scale | Cheaper (30-50% less) | More expensive (convenience tax) |
| Maintenance | You patch, upgrade, monitor | Provider handles it |
| Team needed | 2-3 ML platform engineers | 0-1 engineers |

**Rule of thumb:**
- **< 5 engineers:** Use managed services → save time
- **5-20 engineers:** Mix of managed + self-hosted → balance cost and speed
- **20+ engineers:** Self-hosted platform → cost savings justify the team

### AWS Bedrock vs GCP Vertex AI vs Azure OpenAI

For using LLM APIs (not hosting your own model):

| Service | Models Available | Key Advantage |
|---------|-----------------|---------------|
| **AWS Bedrock** | Claude, Llama, Mistral, Titan | Broadest model selection, AWS integration |
| **GCP Vertex AI** | Gemini, Claude, Llama | Best Gemini integration, multimodal |
| **Azure OpenAI** | GPT-4o, GPT-o1, DALL-E | Direct OpenAI models with enterprise compliance |

### 🤔 When Do I Need My Own GPU Cluster?

Only when:
1. **Data privacy:** Data cannot leave your environment (healthcare, defense)
2. **Cost at scale:** You're spending >$50K/month on API calls
3. **Latency:** You need <50ms response times
4. **Custom models:** You're training/fine-tuning frequently

In [ ]:
# Cell 3 — Managed service decision framework

def recommend_infrastructure(team_size, monthly_budget, data_sensitivity, latency_ms):
    """Recommend infrastructure strategy based on your constraints."""
    recommendations = []
    
    # Team size analysis
    if team_size < 5:
        recommendations.append('🏢 Use MANAGED services (SageMaker/Vertex AI) — your team is too small to maintain K8s')
    elif team_size < 20:
        recommendations.append('🔄 HYBRID approach — managed for experiments, self-hosted for production')
    else:
        recommendations.append('🛠️ BUILD your own platform — team is large enough to justify the investment')
    
    # Budget analysis
    if monthly_budget < 5000:
        recommendations.append('💰 Use SPOT instances for training, API calls for inference')
    elif monthly_budget < 50000:
        recommendations.append('💰 Mix of reserved instances + spot for training')
    else:
        recommendations.append('💰 Consider RESERVED instances (1-year commit) for 30-60% savings')
    
    # Data sensitivity
    if data_sensitivity == 'high':
        recommendations.append('🔒 Self-hosted or VPC-isolated managed services. NO public API calls.')
    elif data_sensitivity == 'medium':
        recommendations.append('🔒 Managed services with VPC endpoints and encryption at rest')
    else:
        recommendations.append('🔒 Standard managed services with default encryption')
    
    # Latency
    if latency_ms < 50:
        recommendations.append('⚡ Self-hosted with local GPU inference (managed services add 20-100ms overhead)')
    elif latency_ms < 200:
        recommendations.append('⚡ Managed inference endpoints (SageMaker/Vertex AI endpoints)')
    else:
        recommendations.append('⚡ API-based inference is fine (Bedrock/Azure OpenAI)')
    
    return recommendations

# Scenario 1: Small startup
print('📋 Scenario 1: AI Startup (3 engineers, $3K/month, public data)')
for r in recommend_infrastructure(3, 3000, 'low', 500):
    print(f'  {r}')

print()
# Scenario 2: Healthcare company
print('📋 Scenario 2: Healthcare AI (15 engineers, $40K/month, patient data)')
for r in recommend_infrastructure(15, 40000, 'high', 100):
    print(f'  {r}')

print()
# Scenario 3: Large tech company
print('📋 Scenario 3: Tech Company (50 engineers, $200K/month, mixed data)')
for r in recommend_infrastructure(50, 200000, 'medium', 30):
    print(f'  {r}')

---

## 4 · Infrastructure as Code (IaC): Never Click Through a Console

### 🤔 What is Infrastructure as Code?

Instead of clicking buttons in the AWS/GCP console to create resources, you write CODE that describes what you want. A tool (Terraform, Pulumi) then creates it for you.

**Analogy:** Clicking through the console is like cooking without a recipe — it works once, but you can't reproduce it. IaC is the recipe: anyone can follow it and get the same result.

| Without IaC | With IaC |
|-------------|----------|
| "I set up the GPU cluster last year... I think I clicked..." | `terraform apply` → identical cluster |
| New team member spends 2 days setting up their environment | `terraform apply` → ready in 10 minutes |
| Disaster recovery: manually recreate everything | `terraform apply` in a new region → done |
| Audit: "who changed the firewall rules?" → unknown | Git log shows exactly who changed what |

### Tool Comparison

| Tool | Language | Multi-Cloud | Best For |
|------|---------|-------------|----------|
| **Terraform** | HCL (declarative) | ✅ Yes | Industry standard, most jobs |
| **Pulumi** | Python, TypeScript | ✅ Yes | Developers who hate learning new languages |
| **CloudFormation** | YAML/JSON | ❌ AWS only | Deep AWS shops |
| **CDK** | Python, TypeScript | ❌ AWS only | AWS + programming skills |

In [ ]:
# Cell 4 — Generate Terraform configuration for AI infrastructure
#
# This generates a REAL Terraform file you could use to provision
# GPU instances, object storage, and a container registry on AWS.

terraform_config = '''
# ============================================
# Terraform: AI Training Infrastructure (AWS)
# ============================================
# Run: terraform init && terraform plan && terraform apply

provider "aws" {
  region = "us-east-1"  # Choose region with GPU availability
}

# --- S3 Bucket for training data and model artifacts ---
resource "aws_s3_bucket" "ml_artifacts" {
  bucket = "my-company-ml-artifacts"
  
  tags = {
    Environment = "production"
    Team        = "ml-platform"
  }
}

# Enable versioning (so you never lose a model checkpoint)
resource "aws_s3_bucket_versioning" "ml_versioning" {
  bucket = aws_s3_bucket.ml_artifacts.id
  versioning_configuration { status = "Enabled" }
}

# --- ECR Repository for ML container images ---
resource "aws_ecr_repository" "ml_service" {
  name = "ml-inference-service"
  image_scanning_configuration { scan_on_push = true }  # Security scanning
}

# --- GPU Instance for training ---
resource "aws_instance" "training_gpu" {
  ami           = "ami-0abcdef1234567890"  # Deep Learning AMI (Ubuntu)
  instance_type = "p4d.24xlarge"            # 8x A100 GPUs
  
  # Use spot for 60-90% savings on training
  instance_market_options {
    market_type = "spot"
    spot_options {
      max_price                      = "15.00"  # Max $/hr you will pay
      spot_instance_type             = "persistent"
      instance_interruption_behavior = "stop"   # Stop (not terminate) on interruption
    }
  }
  
  root_block_device {
    volume_size = 500  # GB — enough for model weights + datasets
    volume_type = "gp3"
  }
  
  tags = {
    Name = "ml-training-gpu"
    Team = "ml-platform"
  }
}

# --- Outputs (values printed after terraform apply) ---
output "s3_bucket" { value = aws_s3_bucket.ml_artifacts.bucket }
output "ecr_repo"  { value = aws_ecr_repository.ml_service.repository_url }
output "gpu_id"    { value = aws_instance.training_gpu.id }
'''

print('🏗️ Terraform Configuration for AI Infrastructure:')
print(terraform_config)
print('Key concepts:')
print('  1. S3 bucket with versioning → never lose model checkpoints')
print('  2. ECR with scan-on-push → security scanning for every image')
print('  3. Spot instance → 60-90% cheaper than on-demand')
print('  4. Tags → track costs by team/project')

---

## 5 · Cloud Cost Management: Don't Go Bankrupt

### 🤔 Why Do AI Teams Overspend?

**Horror story:** A startup left 8 A100 GPUs running over a weekend. Cost: $2,880 in 3 days. For nothing — the training job had finished Friday afternoon.

### The 5 Rules of Cloud Cost Control

| Rule | Implementation | Savings |
|------|---------------|--------|
| **1. Auto-shutdown idle resources** | Lambda function to stop instances with no GPU utilization | 20-40% |
| **2. Use spot/preemptible for training** | Always train with checkpoints on spot | 60-90% |
| **3. Right-size instances** | Don't use H100 when A10G is enough | 50-80% |
| **4. Set budget alerts** | Email at 50%, 80%, 100% of budget | Prevents surprises |
| **5. Tag everything** | Team, project, environment tags on all resources | Accountability |

### ⚖️ Trade-off: Speed vs Cost

| Priority | Strategy | Example |
|----------|----------|--------- |
| Speed first | On-demand H100s, no waiting | Urgent production fix |
| Balanced | Reserved A100s + spot overflow | Regular training pipeline |
| Cost first | Spot instances only, willing to wait | Research experiments |

In [ ]:
# Cell 5 — Cloud cost simulator

def monthly_ai_cost(inference_requests_per_day, training_hours_per_week, 
                     inference_gpu='T4', training_gpu='A100_80GB', use_spot=True):
    """Estimate monthly cloud costs for an AI team."""
    inf_gpu = gpu_catalog[inference_gpu]
    train_gpu = gpu_catalog[training_gpu]
    
    # Inference: assume 100 req/sec per GPU, 24/7
    gpus_for_inference = max(1, inference_requests_per_day // (100 * 3600 * 24) + 1)
    inference_cost = gpus_for_inference * inf_gpu['cost_hr'] * 24 * 30  # 24/7
    
    # Training
    spot_discount = 0.3 if use_spot else 1.0
    training_cost = training_hours_per_week * 4 * train_gpu['cost_hr'] * spot_discount
    
    # Storage (S3/GCS): ~$0.023/GB/month, assume 1TB
    storage_cost = 1000 * 0.023
    
    # Networking: ~10% of compute
    network_cost = (inference_cost + training_cost) * 0.1
    
    total = inference_cost + training_cost + storage_cost + network_cost
    
    return {
        'inference': f'${inference_cost:,.0f}',
        'training': f'${training_cost:,.0f}',
        'storage': f'${storage_cost:,.0f}',
        'network': f'${network_cost:,.0f}',
        'total': f'${total:,.0f}',
    }

print('💰 Monthly Cloud Cost Estimates')
print('=' * 60)

# Small project
small = monthly_ai_cost(10000, 8, 'T4', 'A10G', True)
print(f'\n🏠 Small Project (10K req/day, 8 hrs/week training):')
for k, v in small.items():
    print(f'   {k:<12}: {v}')

# Medium project
medium = monthly_ai_cost(100000, 40, 'A10G', 'A100_80GB', True)
print(f'\n🏢 Medium Project (100K req/day, 40 hrs/week):')
for k, v in medium.items():
    print(f'   {k:<12}: {v}')

# Large project
large = monthly_ai_cost(1000000, 168, 'A100_40GB', 'H100', True)
print(f'\n🏗️ Large Project (1M req/day, 24/7 training):')
for k, v in large.items():
    print(f'   {k:<12}: {v}')

print('\n⚠️ These are rough estimates. Actual costs depend on region, reservations, and usage patterns.')

---

## 🎯 Summary & Key Takeaways

| Concept | What You Learned | Why It Matters |
|---------|-----------------|----------------|
| Cloud Providers | AWS vs GCP vs Azure mapping | Translate skills between clouds |
| GPU Instances | Types, costs, spot pricing | Choose the right GPU for the job |
| Managed vs Self-Hosted | Decision framework | Don't over-engineer or under-invest |
| Infrastructure as Code | Terraform basics | Reproducible, auditable infrastructure |
| Cost Management | 5 rules to prevent overspend | GPU costs can bankrupt a startup |

### 🧠 Key Questions

1. **"Do I NEED cloud for AI?"** → For learning, no. For production, almost always yes.
2. **"Which cloud should I pick?"** → Pick the one your company uses. If starting fresh, AWS has the most jobs, GCP has the best AI tools.
3. **"How do I avoid surprise bills?"** → Budget alerts + auto-shutdown + spot instances + tags.

**Next →** Data Lakehouse -- Open Table Formats for AI Data